In [51]:
import pandas as pd
import geopandas as gpd
import re

Data Validation Utility

In [3]:
def validate_geodataframe(gdf, name="GeoDataFrame"):
    """
    Validate a GeoDataFrame before analysis.
    Checks for null geometries, invalid geometries, and CRS information.
    Optionally auto-fixes invalid geometries using buffer(0).
    
    Parameters:
    -----------
    gdf : geopandas.GeoDataFrame
        The GeoDataFrame to validate
    name : str
        A descriptive name for the dataset (used in print output)
    
    Returns:
    --------
    geopandas.GeoDataFrame
        The validated (and potentially fixed) GeoDataFrame
    """
    print(f"Validating '{name}'...")
    print(f"  - Total records: {len(gdf)}")
    print(f"  - CRS: {gdf.crs}")
    
    null_count = gdf.geometry.isnull().sum()
    invalid_count = (~gdf.geometry.is_valid).sum()
    
    print(f"  - Null geometries: {null_count}")
    print(f"  - Invalid geometries: {invalid_count}")
    
    if null_count > 0:
        print(f"  WARNING: Contains {null_count} null geometries - these rows may cause issues in spatial operations")
    
    if invalid_count > 0:
        print(f"  WARNING: Contains {invalid_count} invalid geometries - attempting to fix with buffer(0)...")
        gdf = gdf.copy()
        gdf['geometry'] = gdf.geometry.buffer(0)
        new_invalid_count = (~gdf.geometry.is_valid).sum()
        if new_invalid_count == 0:
            print(f"  SUCCESS: All geometries are now valid")
        else:
            print(f"  PARTIAL FIX: {new_invalid_count} geometries still invalid")
    
    print(f"  Validation complete.\n")
    return gdf

### 1. HDB resale price with coordinates

1.1 HDB resale price 2025

In [52]:
resale_price_df =  pd.read_csv("../datasets/Resale flat prices based on registration date from Jan-2017 onwards.csv")

In [53]:
def clean_street_name(street):
    if pd.isna(street): return street
    street = street.upper().strip()
    
    # Define common Singapore address abbreviations
    replacements = {
        r'\bBT\b': 'BUKIT',
        r'\bRD\b': 'ROAD',
        r'\bAVE\b': 'AVENUE',
        r'\bDR\b': 'DRIVE',
        r'\bCL\b': 'CLOSE',
        r'\bCTRL\b': 'CENTRAL',
        r'\bCPD\b': 'COMPOUND',
        r'\bCRES\b': 'CRESCENT',
        r'\bGRV\b': 'GROVE',
        r'\bHWY\b': 'HIGHWAY',
        r'\bJLN\b': 'JALAN',
        r'\bKG\b': 'KAMPONG',
        r'\bLN\b': 'LANE', 
        r'\bMT\b': 'MOUNT',
        r'\bPL\b': 'PLACE',
        r'\bRS\b': 'RISE',
        r'\bTER\b': 'TERRACE',
        r'\bTG\b': 'TANJONG',
        r'\bUPP\b': 'UPPER',
        r'\bNTH\b': 'NORTH',
        r'\bSTH\b': 'SOUTH',
        r'\bPK\b': 'PARK',
        r'\bMKT\b': 'MARKET',
        r"\bC'WEALTH\b": 'COMMONWEALTH',
        r"\bLOR\b": 'LORONG',
        r'\bHTS\b': 'HEIGHTS',
        r'\bGDNS\b': 'GARDENS',
        r'\bST\b(?!\.)': 'STREET'
    }
    
    for short, full in replacements.items():
        street = re.sub(short, full, street)
    
    # Remove extra spaces caused by regex
    return " ".join(street.split())

In [ ]:
# filter only 2025 resale data 
resale_price_df["month"]= pd.to_datetime(resale_price_df["month"])
resale_price_2025_df = resale_price_df[resale_price_df["month"].dt.year == 2025].copy()
resale_price_2025_df['street_name'] = resale_price_2025_df['street_name'].apply(clean_street_name)
resale_price_2025_df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
196982,2025-05-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVENUE 10,04 TO 06,44.0,Improved,1979,53 years 01 month,340000.0
196983,2025-08-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVENUE 10,07 TO 09,44.0,Improved,1979,52 years 10 months,353000.0
196984,2025-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVENUE 3,10 TO 12,44.0,Improved,1978,52 years 02 months,318000.0
196985,2025-03-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVENUE 3,10 TO 12,44.0,Improved,1978,51 years 11 months,338000.0
196986,2025-09-01,ANG MO KIO,3 ROOM,308A,ANG MO KIO AVENUE 1,13 TO 15,70.0,Model A,2012,86 years,625000.0


1.2 Existing HDBs

In [59]:
hdb_existing = gpd.read_file("../datasets/HDBExistingBuilding.geojson")
road_namecode = pd.read_excel("../datasets/road_name_code.xlsx", skiprows= 10)

In [60]:
road_namecode = road_namecode[["Domain Value", "Description"]].copy()

# match on "ST_COD" = "Domain Value" to get road description
hdb_existing = hdb_existing.merge(
    road_namecode,
    left_on= "ST_COD", 
    right_on= "Domain Value", 
    how= "left"
)
# ensure all ST_COD has a match
unmatched_rows = hdb_existing[hdb_existing['Domain Value'].isna()]
if unmatched_rows.empty:
    print("All 'ST_COD' values found a match in the road name table.")
else:
    print(f"{len(unmatched_rows)} rows failed to match.")
    print("Missing ST_COD values:")
    print(unmatched_rows['ST_COD'].unique())
# rename and select columns
hdb_existing = hdb_existing.rename(columns={"Description": "STREET_NAME", "geometry": "GEOMETRY"})
hdb_existing = hdb_existing[["BLK_NO", "STREET_NAME", "GEOMETRY"]]
hdb_existing.head()

All 'ST_COD' values found a match in the road name table.


,BLK_NO,STREET_NAME,GEOMETRY
0,208B,CLEMENTI AVENUE 6,"POLYGON ((103.76235 1.32274, 103.76232 1.32274..."
1,238,BUKIT BATOK EAST AVENUE 5,"POLYGON ((103.75446 1.3501, 103.75448 1.35011,..."
2,88,CIRCUIT ROAD,"POLYGON ((103.88589 1.32371, 103.88579 1.3237,..."
3,44A,MARINE CRESCENT,"POLYGON ((103.91325 1.30501, 103.91325 1.30503..."
4,671B,JURONG WEST STREET 65,"POLYGON ((103.70208 1.34397, 103.70208 1.34398..."


1.3 Merge existing HDBs geometry with resale prices

In [63]:
# standardize to string types
resale_price_2025_df['block'] = resale_price_2025_df['block'].astype(str).str.strip()
hdb_existing['BLK_NO'] = hdb_existing['BLK_NO'].astype(str).str.strip()
# match on block and street name to get coordinates
resale_with_geom_df = resale_price_2025_df.merge(
    hdb_existing, 
    left_on=['block', 'street_name'], 
    right_on=['BLK_NO', 'STREET_NAME'], 
    how='left'
)
# rename and select columns
resale_with_geom_df = resale_with_geom_df.rename(columns={'GEOMETRY': 'geometry'})
selected_columns = [
    'month', 
    'town', 
    'flat_type', 
    'block', 
    'street_name', 
    'storey_range', 
    'floor_area_sqm', 
    'flat_model', 
    'lease_commence_date', 
    'remaining_lease', 
    'resale_price', 
    'geometry'
]
resale_with_geom_df = resale_with_geom_df[selected_columns]

In [ ]:
# convert dataframe to geodataframe
resale_with_geom = gpd.GeoDataFrame(resale_with_geom_df, geometry='geometry')
# reproject from geographic CRS (WGS84) to a projected CRS using EPSG:3414 (SVY21)
resale_with_geom_reproj = resale_with_geom.to_crs(3414)
resale_with_geom_reproj.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,geometry
0,2025-05-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVENUE 10,04 TO 06,44.0,Improved,1979,53 years 01 month,340000.0,"POLYGON ((30241.53 38241.555, 30243.683 38240...."
1,2025-08-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVENUE 10,07 TO 09,44.0,Improved,1979,52 years 10 months,353000.0,"POLYGON ((30241.53 38241.555, 30243.683 38240...."
2,2025-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVENUE 3,10 TO 12,44.0,Improved,1978,52 years 02 months,318000.0,"POLYGON ((29817.784 38686.803, 29817.735 38689..."
3,2025-03-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVENUE 3,10 TO 12,44.0,Improved,1978,51 years 11 months,338000.0,"POLYGON ((29817.784 38686.803, 29817.735 38689..."
4,2025-09-01,ANG MO KIO,3 ROOM,308A,ANG MO KIO AVENUE 1,13 TO 15,70.0,Model A,2012,86 years,625000.0,"POLYGON ((29186.232 38608.349, 29186.051 38608..."
